# How Many Basketball Courts Does the Philippines Have Per Capita?
There have been claims that the Philippines has the biggest number of basketball courts per capita, but these have been rarely been backed with data. This notebook tests the domestic half, using live data pulled from OpenStreetMap through the Overpass API.

## Methodology

**Data source:** [OpenStreetMap](https://www.openstreetmap.org/) via the [Overpass API](https://overpass-api.de/api/interpreter).

**Tag categories:**
- **Outdoor courts** — features tagged `leisure=pitch` + `sport=basketball`. OSM's explicit tag for an outdoor basketball court.
- **Covered courts / gyms** — features tagged `leisure=sports_hall`. In the local context, a barangay "covered court" or multi-purpose hall is almost always used as a basketball court by default, even as OSM contributors usually tag the sport value as "multi" rather than "basketball." For this project, every "sports_hall" was counted as a covered court, reported as its own column.

**Geographic scope:**
1. All 17 Metro Manila cities (queried individually.)
2. Philippine total.

**Population:** 
[2024 Census of Population (2024 POPCEN)](https://psa.gov.ph/content/2024-census-population-popcen-population-counts-declared-official-president) — The Philippine Statistics Authority's (PSA) official population count as of 01 July 2024.
- Per-city counts (Metro Manila): `NCR_1.xlsx` https://psa.gov.ph/content/highlights-national-capital-region-ncr-population-2024-census-population-2024-popcen
- Nationwide count: https://psa.gov.ph/content/2024-census-population-popcen-population-counts-declared-official-president.

The 2024 POPCEN only counted population (no housing data so far). It is the current denominator, and reflects the prevailing administrative boundaries, considering a Supreme Court ruling that transferred 10 barangays from Makati City to Taguig City.

- Per-city queries used a bounding box instead of an admin-area lookup per city while the nationwide query used an admin_area lookup, with random delays per request ranging from 15 to 40 seconds.

In [1]:
import requests
import time
import random
import math
import pandas as pd

In [2]:
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

HEADERS = {"User-Agent": "basketball_courts (jonviktor@yahoo.com)"}

# Delay range (seconds)
MIN_DELAY = 15
MAX_DELAY = 40

METRO_MANILA_LGUS = [
    "Quezon City", "Manila", "Makati", "Pasig", "Taguig", "Mandaluyong",
    "Marikina", "Pasay", "Caloocan", "Las Pinas", "Muntinlupa", "Paranaque",
    "Valenzuela", "Malabon", "Navotas", "San Juan", "Pateros",
]

In [3]:
def get_bbox(city_name, retries=4):
    """
    Geocode a Metro Manila City through Nominatim and return an Overpass-style
    bbox tuple (south, west, north, east).
    """
    params = {
        "q": f"{city_name}, Metro Manila, Philippines",
        "format": "json",
        "countrycodes": "ph",
        "limit": 1,
    }
    for attempt in range(retries):
        resp = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=30)
        if resp.status_code == 200 and resp.json():
            result = resp.json()[0]
            south, north, west, east = (float(x) for x in result["boundingbox"])
            print(f"  {city_name} -> matched '{result['display_name']}'")
            return south, west, north, east
        wait = 2 * (attempt + 1)
        print(f"  Nominatim lookup for {city_name} failed (status {resp.status_code}), retrying in {wait}s...")
        time.sleep(wait)
    raise RuntimeError(f"Could not geocode {city_name} via Nominatim")

In [4]:
def run_overpass_query(query, max_retries=6):
    """
    POST a query to the Overpass API, retry on 429 (too many requests) and 504 (server busy).
    """
    for attempt in range(max_retries):
        resp = requests.post(OVERPASS_URL, data={"data": query}, headers=HEADERS, timeout=200)
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code in (429, 504):
            wait = min(120, 10 * (2 ** attempt)) + random.uniform(0, 5)
            print(f"    Overpass returned {resp.status_code}, backing off {wait:.0f}s (attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait)
            continue
        resp.raise_for_status()
    raise RuntimeError(f"Overpass query failed after {max_retries} retries")

In [5]:
def build_city_query(bbox):
    """
    Overpass query counting outdoor pitches and covered sports
    halls within a bounding box.
    """
    south, west, north, east = bbox
    return f"""
    [out:json][timeout:90][bbox:{south},{west},{north},{east}];
    (
      way["leisure"="pitch"]["sport"="basketball"];
      node["leisure"="pitch"]["sport"="basketball"];
      relation["leisure"="pitch"]["sport"="basketball"];
    )->.outdoor;
    (
      way["leisure"="sports_hall"];
      node["leisure"="sports_hall"];
      relation["leisure"="sports_hall"];
    )->.covered;
    .outdoor out count;
    .covered out count;
    """


def build_nationwide_query():
    """
    Build the nationwide Overpass query. 
    """
    return """
    [out:json][timeout:300];
    area["ISO3166-1"="PH"]["admin_level"="2"]->.ph;
    (
      way(area.ph)["leisure"="pitch"]["sport"="basketball"];
      node(area.ph)["leisure"="pitch"]["sport"="basketball"];
      relation(area.ph)["leisure"="pitch"]["sport"="basketball"];
    )->.outdoor;
    (
      way(area.ph)["leisure"="sports_hall"];
      node(area.ph)["leisure"="sports_hall"];
      relation(area.ph)["leisure"="sports_hall"];
    )->.covered;
    .outdoor out count;
    .covered out count;
    """


def parse_counts(response_json):
    """
    Overpass 'out count;' returns each set as a virtual element whose
    tags include a 'total' field.
    """
    elements = response_json.get("elements", [])
    outdoor_total = int(elements[0]["tags"]["total"])
    covered_total = int(elements[1]["tags"]["total"])
    return outdoor_total, covered_total

In [6]:
city_results = []

for i, city in enumerate(METRO_MANILA_LGUS):
    print(f"[{i + 1}/{len(METRO_MANILA_LGUS)}] {city}")
    bbox = get_bbox(city)
    query = build_city_query(bbox)
    data = run_overpass_query(query)
    outdoor, covered = parse_counts(data)
    print(f"  outdoor pitches: {outdoor}, covered courts/gyms: {covered}")
    city_results.append({"city": city, "outdoor_courts": outdoor, "covered_courts": covered})

    if i < len(METRO_MANILA_LGUS) - 1:
        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        print(f"  waiting {delay:.0f}s before next request...\n")
        time.sleep(delay)

print("Done with city-level queries.")

[1/17] Quezon City


  Quezon City -> matched 'Quezon City, Eastern Manila District, Metro Manila, Philippines'


  outdoor pitches: 541, covered courts/gyms: 608
  waiting 24s before next request...



[2/17] Manila


  Manila -> matched 'Manila, Capital District, Metro Manila, Philippines'


...

  San Juan -> matched 'San Juan, Eastern Manila District, Metro Manila, Philippines'


  outdoor pitches: 32, covered courts/gyms: 9
  waiting 28s before next request...



[17/17] Pateros


  Pateros -> matched 'Pateros, Southern Manila District, Metro Manila, Philippines'


  outdoor pitches: 7, covered courts/gyms: 15
Done with city-level queries.


In [7]:
print("Querying nationwide totals for the Philippines...")
time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
nationwide_data = run_overpass_query(build_nationwide_query())
nationwide_outdoor, nationwide_covered = parse_counts(nationwide_data)
print(f"Nationwide -> outdoor pitches: {nationwide_outdoor}, covered courts/gyms: {nationwide_covered}")

Querying nationwide totals for the Philippines...


Nationwide -> outdoor pitches: 11947, covered courts/gyms: 12079


In [8]:
# 2024 Census of Population (2024 POPCEN), sourced directly from the Philippine Statistics Authority
# - Makati and Taguig are boundary-consistent with the 2023 barangay transfer.
# Cross-checked figures, with the sum of 14,001,751 exactly matching the PSA's published total.

POPULATION_2024 = {
    "Quezon City": 3084270,
    "Manila": 1902590,
    "Makati": 309770,
    "Pasig": 853050,
    "Taguig": 1308085,
    "Mandaluyong": 465902,
    "Marikina": 471323,
    "Pasay": 453186,
    "Caloocan": 1712945,
    "Las Pinas": 615549,
    "Muntinlupa": 552225,
    "Paranaque": 703245,
    "Valenzuela": 725173,
    "Malabon": 389929,
    "Navotas": 252878,
    "San Juan": 134312,
    "Pateros": 67319,
}

PHILIPPINES_POPULATION_2024 = 112729484

df = pd.DataFrame(city_results)
df["population_2024"] = df["city"].map(POPULATION_2024)
df

,city,outdoor_courts,covered_courts,population_2024
0,Quezon City,541,608,3084270
1,Manila,129,141,1902590
2,Makati,47,37,309770
3,Pasig,176,123,853050
4,Taguig,264,159,1308085
5,Mandaluyong,51,35,465902
6,Marikina,77,94,471323
7,Pasay,31,33,453186
8,Caloocan,399,469,1712945
9,Las Pinas,285,167,615549


In [9]:
df["total_courts"] = df["outdoor_courts"] + df["covered_courts"]
df["courts_per_100k"] = df["total_courts"] / df["population_2024"] * 100000

nationwide_total = nationwide_outdoor + nationwide_covered
nationwide_row = pd.DataFrame([{
    "city": "PHILIPPINES (nationwide)",
    "outdoor_courts": nationwide_outdoor,
    "covered_courts": nationwide_covered,
    "population_2024": PHILIPPINES_POPULATION_2024,
    "total_courts": nationwide_total,
    "courts_per_100k": nationwide_total / PHILIPPINES_POPULATION_2024 * 100000,
}])

combined = pd.concat([df, nationwide_row], ignore_index=True)
combined = combined[["city", "outdoor_courts", "covered_courts", "total_courts", "population_2024", "courts_per_100k"]]
combined_sorted = combined.sort_values("courts_per_100k", ascending=False).reset_index(drop=True)
combined_sorted

,city,outdoor_courts,covered_courts,total_courts,population_2024,courts_per_100k
0,Las Pinas,285,167,452,615549,73.430385
1,Caloocan,399,469,868,1712945,50.672964
2,Paranaque,200,153,353,703245,50.195878
3,Muntinlupa,139,68,207,552225,37.484721
4,Quezon City,541,608,1149,3084270,37.253548
5,Valenzuela,137,133,270,725173,37.232495
6,Marikina,77,94,171,471323,36.280852
7,Pasig,176,123,299,853050,35.050700
8,Pateros,7,15,22,67319,32.680224
9,Taguig,264,159,423,1308085,32.337348


## Findings

Findings show a single run of notebook and since OSM is edited continuously by volunteers, re-running the notebook later may yield different counts.

Data as of August 1, 2026:

- **Metro Manila**: 2,605 outdoor pitches + 2,353 covered courts/gyms = 4,958 mapped basketball courts across 14.00 million residents — about **35.4 courts per 100,000 residents**.
- **Nationwide**: 11,947 outdoor pitches + 12,079 covered courts/gyms = 24,026 mapped courts across 112.7 million residents — about **21.3 courts per 100,000 residents**. Metro Manila alone accounts for roughly 21% of all OSM-mapped basketball courts in the country while holding about 12% of its population.
- **Highest per-capita city**: Las Piñas (73.4 per 100k), Caloocan (50.7), and Parañaque (50.2).
- **Highest in absolute terms** Quezon City (1,149 total).
- **Lowest per-capita**: Pasay (14.1 per 100k) and Manila (14.2).
- **Outdoor vs. covered split**: across Metro Manila the two categories are close to even (2,605 vs. 2,353), but varies a lot by city.

**Disclaimer::**
- OSM data changes over time, and more tags may mean more contributors than the actual difference in how many courts exist per person.
- This notebook only establishes what OSM has mapped in the Philippines.

In [10]:
combined_sorted.to_csv("basketball_courts_per_capita.csv", index=False)
print("Saved to basketball_courts_per_capita.csv")

Saved to basketball_courts_per_capita.csv


## Mapping courts

Fix coordinates of the courts for visualization purposes.

In [11]:
resp = requests.get(
    NOMINATIM_URL,
    params={"q": "Metro Manila, Philippines", "format": "json", "countrycodes": "ph", "limit": 1},
    headers=HEADERS,
    timeout=30,
)
result = resp.json()[0]
south, north, west, east = (float(x) for x in result["boundingbox"])
metro_bbox = (south, west, north, east)
print("Metro Manila bbox:", metro_bbox, "->", result["display_name"])

Metro Manila bbox: (14.3472554, 120.7917034, 14.7853355, 121.1350232) -> Metro Manila, Philippines


In [12]:
def build_outdoor_points_query(bbox):
    south, west, north, east = bbox
    return f"""
    [out:json][timeout:180][bbox:{south},{west},{north},{east}];
    (
      way["leisure"="pitch"]["sport"="basketball"];
      node["leisure"="pitch"]["sport"="basketball"];
      relation["leisure"="pitch"]["sport"="basketball"];
    );
    out center;
    """


def build_covered_points_query(bbox):
    south, west, north, east = bbox
    return f"""
    [out:json][timeout:180][bbox:{south},{west},{north},{east}];
    (
      way["leisure"="sports_hall"];
      node["leisure"="sports_hall"];
      relation["leisure"="sports_hall"];
    );
    out center;
    """


def elements_to_points(elements, category):
    """Ways/relations only carry a lat/lon under 'center' when queried with
    'out center;'; nodes have lat/lon directly.
    """
    points = []
    for el in elements:
        if el["type"] == "node":
            lat, lon = el["lat"], el["lon"]
        elif "center" in el:
            lat, lon = el["center"]["lat"], el["center"]["lon"]
        else:
            continue
        points.append({
            "category": category,
            "osm_type": el["type"],
            "osm_id": el["id"],
            "name": el.get("tags", {}).get("name"),
            "lat": lat,
            "lon": lon,
        })
    return points

In [13]:
print("Querying outdoor pitch coordinates across Metro Manila...")
outdoor_geo_data = run_overpass_query(build_outdoor_points_query(metro_bbox))
outdoor_points = elements_to_points(outdoor_geo_data["elements"], "outdoor")
print(f"  found {len(outdoor_points)} outdoor pitches")

delay = random.uniform(MIN_DELAY, MAX_DELAY)
print(f"  waiting {delay:.0f}s before next request...\n")
time.sleep(delay)

print("Querying covered court/gym coordinates across Metro Manila...")
covered_geo_data = run_overpass_query(build_covered_points_query(metro_bbox))
covered_points = elements_to_points(covered_geo_data["elements"], "covered")
print(f"  found {len(covered_points)} covered courts/gyms")

all_points = outdoor_points + covered_points
print(f"\nTotal points: {len(all_points)}")

Querying outdoor pitch coordinates across Metro Manila...


  found 1701 outdoor pitches
  waiting 30s before next request...



Querying covered court/gym coordinates across Metro Manila...


    Overpass returned 504, backing off 15s (attempt 1/6)...


    Overpass returned 504, backing off 21s (attempt 2/6)...


  found 1367 covered courts/gyms

Total points: 3068


In [14]:
# Check the data
print(f"Per-city totals (earlier):  outdoor={df['outdoor_courts'].sum()}  covered={df['covered_courts'].sum()}")
print(f"Metro-wide bbox (this run): outdoor={len(outdoor_points)}  covered={len(covered_points)}")

Per-city totals (earlier):  outdoor=2605  covered=2353
Metro-wide bbox (this run): outdoor=1701  covered=1367


In [15]:
import json

geojson_features = [
    {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]},
        "properties": {
            "category": p["category"],
            "osm_type": p["osm_type"],
            "osm_id": p["osm_id"],
            "name": p["name"],
        },
    }
    for p in all_points
]

courts_geojson = {"type": "FeatureCollection", "features": geojson_features}

with open("courts.geojson", "w") as f:
    json.dump(courts_geojson, f)

print(f"Saved {len(geojson_features)} court points to courts.geojson")

Saved 3068 court points to courts.geojson


## Fixing the data:

Counts were inflated due to an overlap. Instead of a bounding rectangle per city, query Overpass against each city's actual OSM administrative boundary.

In [16]:
def get_area_id(city_name, retries=4):
    """Geocode a Metro Manila LGU via Nominatim and return an Overpass area
    id built from the matched OSM way/relation id. Overpass area ids are
    the underlying OSM id plus a fixed offset: +2,400,000,000 for a way,
    +3,600,000,000 for a relation.
    """
    params = {
        "q": f"{city_name}, Metro Manila, Philippines",
        "format": "jsonv2",
        "countrycodes": "ph",
        "limit": 1,
    }
    for attempt in range(retries):
        resp = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=30)
        if resp.status_code == 200 and resp.json():
            result = resp.json()[0]
            osm_type, osm_id = result["osm_type"], int(result["osm_id"])
            offset = {"way": 2400000000, "relation": 3600000000}.get(osm_type)
            if offset is None:
                raise RuntimeError(f"{city_name} resolved to a '{osm_type}', not a way/relation -- can't build an area id")
            print(f"  {city_name} -> matched '{result['display_name']}' ({osm_type}/{osm_id})")
            return offset + osm_id
        wait = 2 * (attempt + 1)
        print(f"  Nominatim lookup for {city_name} failed (status {resp.status_code}), retrying in {wait}s...")
        time.sleep(wait)
    raise RuntimeError(f"Could not geocode {city_name} via Nominatim")


def build_city_area_query(area_id):
    return f"""
    [out:json][timeout:90];
    area({area_id})->.searchArea;
    (
      way["leisure"="pitch"]["sport"="basketball"](area.searchArea);
      node["leisure"="pitch"]["sport"="basketball"](area.searchArea);
      relation["leisure"="pitch"]["sport"="basketball"](area.searchArea);
      way["leisure"="sports_hall"](area.searchArea);
      node["leisure"="sports_hall"](area.searchArea);
      relation["leisure"="sports_hall"](area.searchArea);
    );
    out center;
    """


def classify_and_locate(elements, city):
    """Classify each element back into outdoor/covered using its own OSM
    tags (both categories were queried together in one call), and pull out
    a lat/lon the same way elements_to_points does above.
    """
    points = []
    for el in elements:
        tags = el.get("tags", {})
        if tags.get("leisure") == "pitch" and tags.get("sport") == "basketball":
            category = "outdoor"
        elif tags.get("leisure") == "sports_hall":
            category = "covered"
        else:
            continue
        if el["type"] == "node":
            lat, lon = el["lat"], el["lon"]
        elif "center" in el:
            lat, lon = el["center"]["lat"], el["center"]["lon"]
        else:
            continue
        points.append({
            "city": city,
            "category": category,
            "osm_type": el["type"],
            "osm_id": el["id"],
            "name": tags.get("name"),
            "lat": lat,
            "lon": lon,
        })
    return points

In [17]:
corrected_points = []

for i, city in enumerate(METRO_MANILA_LGUS):
    print(f"[{i + 1}/{len(METRO_MANILA_LGUS)}] {city}")
    area_id = get_area_id(city)
    data = run_overpass_query(build_city_area_query(area_id))
    points = classify_and_locate(data["elements"], city)
    outdoor_n = sum(1 for p in points if p["category"] == "outdoor")
    covered_n = sum(1 for p in points if p["category"] == "covered")
    print(f"  outdoor pitches: {outdoor_n}, covered courts/gyms: {covered_n}")
    corrected_points.extend(points)

    if i < len(METRO_MANILA_LGUS) - 1:
        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        print(f"  waiting {delay:.0f}s before next request...\n")
        time.sleep(delay)

print("Done with corrected per-city queries.")

[1/17] Quezon City


  Quezon City -> matched 'Quezon City, Eastern Manila District, Metro Manila, Philippines' (relation/106569)


    Overpass returned 504, backing off 14s (attempt 1/6)...


  outdoor pitches: 221, covered courts/gyms: 306
  waiting 32s before next request...



[2/17] Manila


...

    Overpass returned 504, backing off 12s (attempt 1/6)...


  outdoor pitches: 21, covered courts/gyms: 7
  waiting 32s before next request...



[17/17] Pateros


  Pateros -> matched 'Pateros, Southern Manila District, Metro Manila, Philippines' (relation/131653)


  outdoor pitches: 3, covered courts/gyms: 10
Done with corrected per-city queries.


In [18]:
pd.DataFrame(corrected_points).head(10)

,city,category,osm_type,osm_id,name,lat,lon
0,Quezon City,outdoor,node,1113105183,NaN,14.674009,121.032948
1,Quezon City,outdoor,node,2607305775,NaN,14.713112,121.096506
2,Quezon City,outdoor,node,4821778725,Casa Milan Basketball Court,14.721580,121.057484
3,Quezon City,outdoor,node,5467579421,NaN,14.666172,121.045048
4,Quezon City,outdoor,node,5893744807,NaN,14.621536,121.042680
5,Quezon City,outdoor,node,7001382555,Don Enrique Heights Covered Basket Ball Court,14.684062,121.081611
6,Quezon City,outdoor,node,7864758717,NaN,14.703972,121.052840
7,Quezon City,outdoor,node,8370321689,NaN,14.626121,121.033940
8,Quezon City,outdoor,node,10074432333,Villa Hermano 4 Basketball Court,14.702578,121.048156
9,Quezon City,outdoor,node,10125334944,NaN,14.710718,121.090953


In [19]:
from collections import defaultdict

tally = defaultdict(lambda: {"outdoor_courts": 0, "covered_courts": 0})
for p in corrected_points:
    key = "outdoor_courts" if p["category"] == "outdoor" else "covered_courts"
    tally[p["city"]][key] += 1

city_results_corrected = [{"city": city, **tally[city]} for city in METRO_MANILA_LGUS]

comparison = pd.DataFrame(city_results_corrected).merge(
    df[["city", "outdoor_courts", "covered_courts"]], on="city", suffixes=("_corrected", "_bbox")
)
comparison

,city,outdoor_courts_corrected,covered_courts_corrected,outdoor_courts_bbox,covered_courts_bbox
0,Quezon City,221,306,541,608
1,Manila,92,113,129,141
2,Makati,21,12,47,37
3,Pasig,89,69,176,123
4,Taguig,125,73,264,159
5,Mandaluyong,36,29,51,35
6,Marikina,43,65,77,94
7,Pasay,11,17,31,33
8,Caloocan,114,137,399,469
9,Las Pinas,71,65,285,167


In [20]:
df_corrected = pd.DataFrame(city_results_corrected)
df_corrected["population_2024"] = df_corrected["city"].map(POPULATION_2024)
df_corrected["total_courts"] = df_corrected["outdoor_courts"] + df_corrected["covered_courts"]
df_corrected["courts_per_100k"] = df_corrected["total_courts"] / df_corrected["population_2024"] * 100000

nationwide_row_corrected = pd.DataFrame([{
    "city": "PHILIPPINES (nationwide)",
    "outdoor_courts": nationwide_outdoor,
    "covered_courts": nationwide_covered,
    "population_2024": PHILIPPINES_POPULATION_2024,
    "total_courts": nationwide_outdoor + nationwide_covered,
    "courts_per_100k": (nationwide_outdoor + nationwide_covered) / PHILIPPINES_POPULATION_2024 * 100000,
}])

combined_corrected = pd.concat([df_corrected, nationwide_row_corrected], ignore_index=True)
combined_corrected = combined_corrected[["city", "outdoor_courts", "covered_courts", "total_courts", "population_2024", "courts_per_100k"]]
combined_corrected_sorted = combined_corrected.sort_values("courts_per_100k", ascending=False).reset_index(drop=True)
combined_corrected_sorted

,city,outdoor_courts,covered_courts,total_courts,population_2024,courts_per_100k
0,Paranaque,93,83,176,703245,25.026840
1,Marikina,43,65,108,471323,22.914222
2,Las Pinas,71,65,136,615549,22.094098
3,PHILIPPINES (nationwide),11947,12079,24026,112729484,21.312969
4,San Juan,21,7,28,134312,20.846983
5,Pateros,3,10,13,67319,19.311041
6,Pasig,89,69,158,853050,18.521775
7,Muntinlupa,71,28,99,552225,17.927475
8,Quezon City,221,306,527,3084270,17.086701
9,Valenzuela,44,67,111,725173,15.306692


In [21]:
combined_corrected_sorted.to_csv("basketball_courts_per_capita.csv", index=False)

geojson_features_corrected = [
    {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]},
        "properties": {
            "city": p["city"],
            "category": p["category"],
            "osm_type": p["osm_type"],
            "osm_id": p["osm_id"],
            "name": p["name"],
        },
    }
    for p in corrected_points
]
courts_geojson_corrected = {"type": "FeatureCollection", "features": geojson_features_corrected}

with open("courts.geojson", "w") as f:
    json.dump(courts_geojson_corrected, f)

print(f"Saved corrected basketball_courts_per_capita.csv and {len(geojson_features_corrected)} court points to courts.geojson")

Saved corrected basketball_courts_per_capita.csv and 2192 court points to courts.geojson


## Corrected findings

- **Metro Manila corrected total**: 1,084 outdoor pitches + 1,108 covered courts/gyms = **2,192 mapped basketball courts** across 14.00 million residents — about **15.7 courts per 100,000 residents**. That's less than *half* the original (inflated) estimate of 35.4 per 100k.
- **Nationwide stays unchanged at 21.3 per 100k** (that query was never bbox-based, so it was never affected by the overlap bug).
- **The headline finding flips**: Metro Manila's per-capita rate (15.7) is now *below* the nationwide rate (21.3), not comfortably above it. Only **3 of the 17 LGUs** — Parañaque (25.0), Marikina (22.9), and Las Piñas (22.1) — exceed the national rate at all; San Juan (20.8) comes close. The other 13 cities, including Quezon City, all sit below it.
- **Quezon City** is still the largest in absolute terms (527 courts — 221 outdoor + 306 covered) but now ranks 9th of 17 per capita (17.1 per 100k), squarely below the national average.
- **Lowest per-capita**: Navotas (4.3 per 100k) and Pasay (6.2) — both were mid-pack in the flawed version, which shows how much the bbox-overlap bug was distorting small, geographically "squeezed" cities specifically (their tiny rectangles happened to swallow a disproportionate share of bigger neighbors' courts).
- **The original ranking essentially inverted.** Las Piñas, previously the runaway #1 at 73.4 per 100k, drops to 3rd at 22.1 — a real number, but nowhere near as extreme once double-counted neighboring courts are removed. Taguig and Caloocan, previously in the top 4, now sit in the bottom half.

## Nationwide court coordinates

In [22]:
def build_nationwide_points_query():
    return """
    [out:json][timeout:300];
    area["ISO3166-1"="PH"]["admin_level"="2"]->.ph;
    (
      way["leisure"="pitch"]["sport"="basketball"](area.ph);
      node["leisure"="pitch"]["sport"="basketball"](area.ph);
      relation["leisure"="pitch"]["sport"="basketball"](area.ph);
      way["leisure"="sports_hall"](area.ph);
      node["leisure"="sports_hall"](area.ph);
      relation["leisure"="sports_hall"](area.ph);
    );
    out center;
    """


def classify_and_locate_nationwide(elements):
    """Same tag-based classification as classify_and_locate above, minus the
    per-city label (nationwide points aren't queried per-LGU, so there's no
    city to attach without a separate reverse-geocoding step).
    """
    points = []
    for el in elements:
        tags = el.get("tags", {})
        if tags.get("leisure") == "pitch" and tags.get("sport") == "basketball":
            category = "outdoor"
        elif tags.get("leisure") == "sports_hall":
            category = "covered"
        else:
            continue
        if el["type"] == "node":
            lat, lon = el["lat"], el["lon"]
        elif "center" in el:
            lat, lon = el["center"]["lat"], el["center"]["lon"]
        else:
            continue
        points.append({"category": category, "lat": lat, "lon": lon})
    return points


print("Querying nationwide court coordinates (single request, ~40-60s)...")
nationwide_geo_data = run_overpass_query(build_nationwide_points_query())
nationwide_points = classify_and_locate_nationwide(nationwide_geo_data["elements"])
print(f"Got {len(nationwide_points)} nationwide court points "
      f"({sum(1 for p in nationwide_points if p['category'] == 'outdoor')} outdoor, "
      f"{sum(1 for p in nationwide_points if p['category'] == 'covered')} covered)")

Querying nationwide court coordinates (single request, ~40-60s)...


Got 24027 nationwide court points (11947 outdoor, 12080 covered)


In [23]:
nationwide_geojson_features = [
    {
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [p["lon"], p["lat"]]},
        "properties": {"category": p["category"]},
    }
    for p in nationwide_points
]

nationwide_courts_geojson = {"type": "FeatureCollection", "features": nationwide_geojson_features}

with open("nationwide_courts.geojson", "w") as f:
    json.dump(nationwide_courts_geojson, f)

print(f"Saved {len(nationwide_geojson_features)} points to nationwide_courts.geojson")

Saved 24027 points to nationwide_courts.geojson


## Courts per square kilometer

In [24]:
def ring_area_m2(ring, lat0):
    """Shoelace formula on an equirectangular projection centered at lat0.
    Fine for city-sized polygons; not meant for anything continental.
    """
    R = 6371000
    lat0_rad = math.radians(lat0)
    pts = [(math.radians(lon) * R * math.cos(lat0_rad), math.radians(lat) * R) for lon, lat in ring]
    area = 0
    n = len(pts)
    for i in range(n):
        x1, y1 = pts[i]
        x2, y2 = pts[(i + 1) % n]
        area += x1 * y2 - x2 * y1
    return abs(area) / 2


def polygon_area_km2(geometry):
    """Handles both Polygon and MultiPolygon (Caloocan and Las Pinas are
    MultiPolygon in this dataset); each ring's area is summed regardless of
    exterior/hole, since none of these 17 city polygons have holes.
    """
    rings = geometry["coordinates"] if geometry["type"] == "Polygon" else [
        ring for polygon in geometry["coordinates"] for ring in polygon
    ]
    lat0 = rings[0][0][1]
    return sum(ring_area_m2(ring, lat0) for ring in rings) / 1e6


with open("metro_manila_lgus.geojson") as f:
    lgu_geojson = json.load(f)

LAND_AREA_KM2 = {
    feat["properties"]["city"]: polygon_area_km2(feat["geometry"])
    for feat in lgu_geojson["features"]
}

for city, area in sorted(LAND_AREA_KM2.items(), key=lambda kv: -kv[1]):
    print(f"{city:15s} {area:6.2f} km2")

Quezon City     163.09 km2
Caloocan         53.41 km2
Valenzuela       46.71 km2
Paranaque        44.98 km2
Manila           42.10 km2
Muntinlupa       39.30 km2
Taguig           35.66 km2
Las Pinas        32.85 km2
Pasig            31.38 km2
Makati           25.05 km2
Marikina         23.00 km2
Pasay            17.95 km2
Malabon          15.84 km2
Mandaluyong      11.38 km2
Navotas          10.54 km2
San Juan          5.78 km2
Pateros           1.61 km2


In [ ]:
PHILIPPINES_LAND_AREA_KM2 = 298170

density = combined_corrected_sorted.copy()
density["land_area_km2"] = density["city"].map(LAND_AREA_KM2)
density.loc[density["city"] == "PHILIPPINES (nationwide)", "land_area_km2"] = PHILIPPINES_LAND_AREA_KM2
density["courts_per_km2"] = density["total_courts"] / density["land_area_km2"]

density_sorted = density.sort_values("courts_per_km2", ascending=False).reset_index(drop=True)
density_sorted[["city", "total_courts", "land_area_km2", "courts_per_km2", "courts_per_100k"]]

In [26]:
# Bake courts_per_km2 into the choropleth source too, and refresh the CSV
# export so both density metrics are available side by side.
for feat in lgu_geojson["features"]:
    city = feat["properties"]["city"]
    row = density[density["city"] == city].iloc[0]
    feat["properties"]["land_area_km2"] = round(row["land_area_km2"], 2)
    feat["properties"]["courts_per_km2"] = round(row["courts_per_km2"], 2)

with open("metro_manila_lgus.geojson", "w") as f:
    json.dump(lgu_geojson, f)

density_sorted.to_csv("basketball_courts_per_capita.csv", index=False)
print("Updated metro_manila_lgus.geojson and basketball_courts_per_capita.csv with courts_per_km2")

Updated metro_manila_lgus.geojson and basketball_courts_per_capita.csv with courts_per_km2


## Density findings: a different ranking entirely

Switching from "per resident" to "per square kilometer" reshuffles the list — it isn't just a rescaled version of the per-capita ranking:

- **Pateros** tops the density ranking (8.05 courts/km²) simply by being tiny (1.6 km²) with a modest but non-negligible court count (13). It was mid-pack on per-capita (19.31/100k).
- **Mandaluyong** is the standout mover: **13.95 per 100k residents (near the bottom of the per-capita ranking)** but **5.71 courts/km² (2nd overall on density)**. It's not that Mandaluyong has few courts relative to its land — it's small and genuinely court-dense — it's that it also has a lot of *people* per court, since it's one of the most densely populated cities in the region. Per-capita and per-area are answering two different questions, and Mandaluyong is the clearest case where they disagree.
- **Quezon City**, the runaway leader in absolute courts (527) and solidly mid-pack per capita, drops to 11th on density (3.23/km²) — it's simply too large in land area for its court count to look dense.
- **Navotas, Makati, and Pasay** rank lowest on both metrics — genuinely court-scarce, not just an artifact of population or land-area scaling.
- **Nationwide**, courts/km² is naturally tiny (0.08 courts/km², using PSA's 298,170 km² official land area) compared to any individual Metro Manila LGU (1.0–8.1/km²) — most of the country's land is rural or unmapped for this purpose, so this comparison mostly restates that Metro Manila is far more urbanized, not that it has a meaningfully different "true" density of courts relative to livable/built space.

**Takeaway:** land-area density is a legitimate second lens, not just a curiosity — it surfaces cities like Mandaluyong that the per-capita ranking undersells, precisely because it isolates geography from population. Which metric is "right" depends on the question: per-capita is the better proxy for an individual resident's access to a court; per-km² is the better proxy for how visually/physically saturated a city's land is with courts.